In [1]:
import requests
import re
import wikitextparser as wtp
import xml.etree.ElementTree as ET
import pandas as pd
import spacy
import urllib
import logging

default_headers = {
  # 'Authorization': 'Bearer ,
  'User-Agent': 'WatermarkingLLM (jeremi.ochab@uj.edu.pl)'
}

session = requests.Session()
session.headers.update(default_headers)

nlp = spacy.load("en_core_web_lg")
sent_max = 10

In [20]:
# Wikimedia REST API
# https://en.wikipedia.org/api/rest_v1/#/Page%20content/get_page_random__format_
# Get random title and its latest revision ID

def get_rand_title(session):
    url = 'https://en.wikipedia.org/api/rest_v1/page/random/title'
    response = session.get(url)
    data = response.json()
    logging.info(f'{data}')
    if 'items' in data.keys():
        random_title = data['items'][0]['title']
        random_revid = data['items'][0]['rev']
        return random_title, random_revid
    else:
        return False, False

# MediaWiki ACTION API
# https://www.mediawiki.org/wiki/API:Query#GET_request
# Get revision summary
def get_rev_summary(session,article_title, revision_id):
    # https://docs.python.org/3/library/urllib.parse.html#parsing-ascii-encoded-bytes
    url = f'https://en.wikipedia.org/w/api.php?action=query&format=json&prop=revisions&rvprop=content&rvsection=0&rvstartid={revision_id}&titles={urllib.parse.quote_plus(article_title)}'
    response = session.get(url)
    data = response.json()
    pages = data['query']['pages']
    page_id = next(iter(pages))
    content = pages[page_id]['revisions'][0]
    if '*' in content.keys():
        content = content['*']
        return content
    else:
        return False

def get_rev_id(session, article_title, older_than_date = '2021'):
    params = {
        'action': 'query',
        'format': 'json',
        'prop': 'revisions',
        'rvprop': 'ids|timestamp',
        'rvlimit':1,
        # 'rvdir':'older',
        'rvstart':older_than_date+'-12-31T00:00:00Z',
        'titles': urllib.parse.quote_plus(article_title),
    }
    
    # Make the API request
    response = session.get('https://en.wikipedia.org/w/api.php', params=params)
    data = response.json()
    # Get revision ID and timestamp
    for page_id, page_info in data['query']['pages'].items():
        # Check if the 'revisions' key exists in the page_info
        if 'revisions' in page_info:
            revisions = page_info['revisions'][0]
            return revisions
        else:
            False

def get_rev_xml(session,revid):
    params = {
        'action': 'parse',
        'format': 'json',
        'prop': 'parsetree', #wikitext|text|parsetree',
        'section': 0,
        'oldid': revid,
    }
    
    # Make the API request
    response = session.get('https://en.wikipedia.org/w/api.php', params=params)
    data = response.json()
    if response.status_code == 400 or 'error'in data.keys():
        return False
    # Strip XML tags and their content
    xml = data['parse']['parsetree']['*']
    root = ET.fromstring(xml)
    if root.text:
        tails = root.text
    else:
        tails = ''
    for child in root:
        if child.tail is not None:
            tails += child.tail
    return tails

def check_sentences(plain_text_summary, sent_max = 10):
    doc = nlp(plain_text_summary)
    sents = [sent.text for sent in doc.sents]
    if len(sents)<sent_max:
        return False
    else:
        return True

def plain_text(wikitext):
    # Parse the wikitext
    parsed = wtp.parse(wikitext)
    
    # Remove references
    spans = [tag.span for tag in parsed.get_tags()]+[tag.span for tag in parsed.get_tables()]+[p.span for p in parsed.wikilinks if re.search(r'^File:',p.target)]
    spans.sort(reverse=True)
    for start_index, end_index in spans:
        wikitext = wikitext[:start_index] + wikitext[end_index:]
    
    parsed = wtp.parse(wikitext)
    plaintext = parsed.plain_text()
    plaintext = re.sub(r'[\(\[][\)\]]', '', plaintext)
    plaintext = re.sub('\xa0', ' ', plaintext) # spacje niełamiące
    plaintext = re.sub(r'[ ]{2}', ' ', plaintext) # spacje wielokrotne
    plaintext = re.sub(r' ,', ',', plaintext) # spacje pozostałe po usuniętych templatach
    plaintext = re.sub(r'\([ ,]+', '(', plaintext) # spacje pozostałe po usuniętych templatach
    plaintext = plaintext.replace('\n\n',' ').strip() # zostawia wyliczenia, np. gdy jest \n* item 1\n* item 2 ...
    # print(plaintext)
    return plaintext, parsed

In [21]:
import time
import logging
logging.basicConfig(level=logging.DEBUG, filename="wiki_logfile4", filemode="a+", format="%(asctime)-15s %(levelname)-8s %(message)s")

wikipedia_data = pd.DataFrame(columns=['term','text','wikitext','timestamp'])

while len(wikipedia_data) < 600:
    random_title, random_revid = get_rand_title(session)
    # Gdyby losowane były powtarzające się, losuj aż znajdziesz nowe
    while random_title in list(wikipedia_data['term']):
        time.sleep(.2)
        random_title, random_revid = get_rand_title(session)
    if random_title:
        random_rev = get_rev_id(session, random_title)
        if random_rev:
            revid = random_rev['revid']
            random_summary = get_rev_xml(session,revid)
            if not random_summary:
                logging.info(f'Term {len(wikipedia_data)}: no summary in the revision.')
            else:
                plain_text_summary, parsed_summary = plain_text(random_summary)
                
                if check_sentences(plain_text_summary):
                    wikipedia_row = {'term': random_title, 'text': plain_text_summary, 'wikitext': parsed_summary, 'timestamp': random_rev['timestamp']}
                    wikipedia_data = pd.concat([wikipedia_data, pd.DataFrame([wikipedia_row])], ignore_index=True)
                    pd.DataFrame([wikipedia_row]).to_csv("wikipedia_summaries4.csv", mode='a', header=False)
                    logging.info(f'Term {len(wikipedia_data)}: {random_title}')
                else:
                    logging.info(f'Term {len(wikipedia_data)}: summary too short')
        else:
            logging.info(f'Term {len(wikipedia_data)}: revision too recent')
    else:
        logging.info(f'Term {len(wikipedia_data)}: bad request')
        
    # Delay wiki queries
    time.sleep(.1)

# wikipedia_data.to_csv("wikipedia_summaries3.csv", mode='a', header=False)

KeyboardInterrupt: 

In [ ]:
session.close()